# Lab 7: Neural Networks from Scratch

**ECON5129 Statistical Machine Learning** &middot; Adam Smith Business School, University of Glasgow

{{COLAB_BADGE}}

This lab accompanies Lecture 7. By the end of the session you should be able to:

1. Describe a neural network as a basis expansion whose basis functions are estimated rather than chosen.
2. Implement the forward pass, the backward pass and gradient descent in numpy, and verify the gradients numerically.
3. Compare stochastic gradient descent, momentum and Adam on the same problem.
4. Apply weight decay, early stopping and dropout, and say which does what.
5. Reproduce the double descent curve and explain why it does not contradict the bias-variance trade-off of Lecture 3.

Everything in this lab runs in numpy on a laptop. No deep learning library is needed, and none is used until Lab 8.

**Plan for the session**

| Time | Part |
|---|---|
| 0:00 - 0:20 | Part 1. A network is a learned basis |
| 0:20 - 0:50 | Part 2. Backpropagation, written out |
| 0:50 - 1:10 | Part 3. Optimisers |
| 1:10 - 1:35 | Part 4. Regularisation |
| 1:35 - 2:00 | Part 5. Overparameterisation and double descent |

## Setup

In [ ]:
import os
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/korobilis/ECON5129-labs/main"

if not os.path.exists("econ5129_utils.py"):
    urllib.request.urlretrieve(f"{REPO_RAW}/econ5129_utils.py", "econ5129_utils.py")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import econ5129_utils as e5

e5.set_style()
rng = np.random.default_rng(5129)

## Part 1. A network is a learned basis

Lab 5 fitted $f(x) = \sum_m \beta_m \phi_m(x)$ with basis functions $\phi_m$ chosen in advance: monomials, radial bumps, splines. A single hidden layer network has the same form,

$$ f(\mathbf{x}) = \beta_0 + \sum_{m=1}^{M} \beta_m \, \sigma\!\left(w_{m0} + \mathbf{w}_m'\mathbf{x}\right), $$

with one crucial difference: the weights $\mathbf{w}_m$ inside the activation function are estimated too. The basis adapts to the data instead of being fixed by the analyst.

Start with the activation functions themselves.

In [ ]:
z = np.linspace(-4, 4, 400)
activations = {
    "sigmoid": 1 / (1 + np.exp(-z)),
    "tanh": np.tanh(z),
    "ReLU": np.maximum(z, 0.0),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, values in activations.items():
    axes[0].plot(z, values, label=name)
axes[0].set_title("Activation functions")
axes[0].legend()

derivatives = {
    "sigmoid": activations["sigmoid"] * (1 - activations["sigmoid"]),
    "tanh": 1 - np.tanh(z) ** 2,
    "ReLU": (z > 0).astype(float),
}
for name, values in derivatives.items():
    axes[1].plot(z, values, label=name)
axes[1].set_title("Their derivatives, which is what backpropagation multiplies")
axes[1].legend()
fig.tight_layout()
plt.show()

The derivative panel explains the historical shift to ReLU. Sigmoid and tanh derivatives vanish away from zero, so in a deep network the chain rule multiplies many small numbers together and the gradient reaching the early layers disappears. The ReLU derivative is either zero or one.

To see that the hidden units really are basis functions, fit a network with fixed random inner weights, so that only the output layer is estimated. That is exactly a linear regression on random features.

In [ ]:
def relu(a):
    return np.maximum(a, 0.0)


def f_target(x):
    return np.sin(3 * x) + 0.3 * x ** 2


n, noise = 200, 0.25
x = np.sort(rng.uniform(-3, 3, n))
y = f_target(x) + noise * rng.normal(size=n)
grid = np.linspace(-3.2, 3.2, 400)

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, M in zip(axes, [3, 10, 60]):
    W = rng.normal(scale=1.5, size=M)
    b = rng.normal(scale=2.0, size=M)

    H_train = relu(x[:, None] * W + b)
    H_grid = relu(grid[:, None] * W + b)

    coef, *_ = np.linalg.lstsq(np.column_stack([np.ones(n), H_train]), y, rcond=None)
    fit = np.column_stack([np.ones(len(grid)), H_grid]) @ coef

    ax.scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.5)
    ax.plot(grid, f_target(grid), color=e5.COLORS[0], label="truth")
    ax.plot(grid, fit, color=e5.COLORS[2], label="fit")
    ax.set_title(f"{M} random hidden units")
axes[0].legend(fontsize=9)
fig.suptitle("Random features: only the output weights are estimated")
fig.tight_layout()
plt.show()

## Part 2. Backpropagation, written out

Training the inner weights requires their gradients. For a network with one hidden layer, squared error loss, and $\mathbf{a} = \mathbf{X}\mathbf{W}_1 + \mathbf{b}_1$, $\mathbf{h} = \sigma(\mathbf{a})$, $\widehat{\mathbf{y}} = \mathbf{h}\mathbf{w}_2 + b_2$:

$$ \frac{\partial L}{\partial \widehat{y}} = \frac{2}{n}(\widehat{y} - y), \qquad \frac{\partial L}{\partial \mathbf{w}_2} = \mathbf{h}'\delta_2, \qquad \delta_1 = (\delta_2 \mathbf{w}_2') \odot \sigma'(\mathbf{a}), \qquad \frac{\partial L}{\partial \mathbf{W}_1} = \mathbf{X}'\delta_1 . $$

Backpropagation is the chain rule applied in the order that reuses computations. The implementation below handles any number of layers.

In [ ]:
class NeuralNetwork:
    """Fully connected feedforward network with ReLU hidden units, trained by Adam."""

    def __init__(self, layer_sizes, seed=0, weight_decay=0.0, dropout=0.0):
        gen = np.random.default_rng(seed)
        self.W, self.b = [], []
        for n_in, n_out in zip(layer_sizes[:-1], layer_sizes[1:]):
            self.W.append(gen.normal(scale=np.sqrt(2.0 / n_in), size=(n_in, n_out)))
            self.b.append(np.zeros(n_out))
        self.weight_decay = weight_decay
        self.dropout = dropout
        self.generator = gen

    def forward(self, X, training=False):
        """Return the output and the intermediate quantities needed for the backward pass."""
        activations, pre_activations, masks = [X], [], []
        A = X
        for layer, (W, b) in enumerate(zip(self.W, self.b)):
            Z = A @ W + b
            pre_activations.append(Z)
            if layer < len(self.W) - 1:
                A = relu(Z)
                if training and self.dropout > 0:
                    mask = (self.generator.random(A.shape) > self.dropout) / (1 - self.dropout)
                    A = A * mask
                    masks.append(mask)
                else:
                    masks.append(None)
            else:
                A = Z
            activations.append(A)
        return A.ravel(), activations, pre_activations, masks

    def gradients(self, X, y):
        """Analytic gradients of the mean squared error loss."""
        out, activations, pre_activations, masks = self.forward(X, training=True)
        n = len(y)

        grads_W = [np.zeros_like(W) for W in self.W]
        grads_b = [np.zeros_like(b) for b in self.b]

        delta = (2.0 / n) * (out - y)[:, None]
        for layer in reversed(range(len(self.W))):
            grads_W[layer] = activations[layer].T @ delta + 2 * self.weight_decay * self.W[layer]
            grads_b[layer] = delta.sum(axis=0)
            if layer > 0:
                delta = delta @ self.W[layer].T
                if masks[layer - 1] is not None:
                    delta = delta * masks[layer - 1]
                delta = delta * (pre_activations[layer - 1] > 0)
        return grads_W, grads_b

    def loss(self, X, y):
        out, *_ = self.forward(X, training=False)
        penalty = self.weight_decay * sum(np.sum(W ** 2) for W in self.W)
        return float(np.mean((out - y) ** 2) + penalty)

    def predict(self, X):
        out, *_ = self.forward(X, training=False)
        return out

    def fit(self, X, y, n_epochs=400, batch_size=32, learning_rate=0.01,
            X_val=None, y_val=None, patience=None):
        """Train by Adam, optionally with early stopping on a validation set."""
        mW = [np.zeros_like(W) for W in self.W]
        vW = [np.zeros_like(W) for W in self.W]
        mb = [np.zeros_like(b) for b in self.b]
        vb = [np.zeros_like(b) for b in self.b]
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        step = 0

        history = {"train": [], "validation": []}
        best_loss, best_state, wait = np.inf, None, 0

        for epoch in range(n_epochs):
            order = self.generator.permutation(len(y))
            for start in range(0, len(y), batch_size):
                idx = order[start:start + batch_size]
                grads_W, grads_b = self.gradients(X[idx], y[idx])
                step += 1
                for layer in range(len(self.W)):
                    mW[layer] = beta1 * mW[layer] + (1 - beta1) * grads_W[layer]
                    vW[layer] = beta2 * vW[layer] + (1 - beta2) * grads_W[layer] ** 2
                    mb[layer] = beta1 * mb[layer] + (1 - beta1) * grads_b[layer]
                    vb[layer] = beta2 * vb[layer] + (1 - beta2) * grads_b[layer] ** 2

                    mW_hat = mW[layer] / (1 - beta1 ** step)
                    vW_hat = vW[layer] / (1 - beta2 ** step)
                    mb_hat = mb[layer] / (1 - beta1 ** step)
                    vb_hat = vb[layer] / (1 - beta2 ** step)

                    self.W[layer] -= learning_rate * mW_hat / (np.sqrt(vW_hat) + eps)
                    self.b[layer] -= learning_rate * mb_hat / (np.sqrt(vb_hat) + eps)

            history["train"].append(self.loss(X, y))
            if X_val is not None:
                val_loss = self.loss(X_val, y_val)
                history["validation"].append(val_loss)
                if patience is not None:
                    if val_loss < best_loss - 1e-8:
                        best_loss, wait = val_loss, 0
                        best_state = ([W.copy() for W in self.W], [b.copy() for b in self.b])
                    else:
                        wait += 1
                        if wait >= patience:
                            self.W, self.b = best_state
                            history["stopped_at"] = epoch + 1
                            break
        return history

### Checking the gradients

Analytic gradients are easy to get wrong and the errors are silent: the network trains, just badly. Always compare against a numerical derivative before trusting an implementation.

In [ ]:
net_check = NeuralNetwork([1, 6, 4, 1], seed=1)
X_check = x[:20, None]
y_check = y[:20]

grads_W, grads_b = net_check.gradients(X_check, y_check)

eps = 1e-6
layer = 1
i, j = np.unravel_index(np.argmax(np.abs(grads_W[layer])), grads_W[layer].shape)
original = net_check.W[layer][i, j]

net_check.W[layer][i, j] = original + eps
loss_plus = net_check.loss(X_check, y_check)
net_check.W[layer][i, j] = original - eps
loss_minus = net_check.loss(X_check, y_check)
net_check.W[layer][i, j] = original

numerical = (loss_plus - loss_minus) / (2 * eps)
print(f"analytic gradient  : {grads_W[layer][i, j]:.10f}")
print(f"numerical gradient : {numerical:.10f}")
print(f"relative difference: {abs(numerical - grads_W[layer][i, j]) / abs(numerical):.2e}")

### Exercise 1

One coefficient is not a test. Check them all.

1. Write `gradient_check(net, X, y)` that loops over every weight and bias, computes the two-sided numerical derivative, and returns the largest relative discrepancy.
2. Run it on a network with layers `[1, 6, 4, 1]`.
3. Then deliberately break the implementation: change the ReLU derivative in `gradients` from `> 0` to `>= 0` and rerun. Does the check catch it? Should it?

In [ ]:
def gradient_check(net, X, y, eps=1e-6):  #@keep
    """Largest relative difference between analytic and numerical gradients."""  #@keep
    grads_W, grads_b = net.gradients(X, y)
    worst = 0.0

    for layer in range(len(net.W)):
        for params, grads in [(net.W[layer], grads_W[layer]), (net.b[layer], grads_b[layer])]:
            it = np.nditer(params, flags=["multi_index"])
            while not it.finished:
                idx = it.multi_index
                original = params[idx]
                params[idx] = original + eps
                plus = net.loss(X, y)
                params[idx] = original - eps
                minus = net.loss(X, y)
                params[idx] = original

                numeric = (plus - minus) / (2 * eps)
                denominator = max(abs(numeric), 1e-8)
                worst = max(worst, abs(numeric - grads[idx]) / denominator)
                it.iternext()
    return worst


net_full = NeuralNetwork([1, 6, 4, 1], seed=2)
print(f"largest relative gradient error: {gradient_check(net_full, X_check, y_check):.2e}")

Now train the network properly and compare it with the random feature fit from Part 1.

In [ ]:
X_1d = x[:, None]
net = NeuralNetwork([1, 32, 32, 1], seed=3)
history = net.fit(X_1d, y, n_epochs=600, batch_size=32, learning_rate=0.01)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["train"])
axes[0].set_yscale("log")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("training loss")
axes[0].set_title("Training curve")

axes[1].scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.5)
axes[1].plot(grid, f_target(grid), color=e5.COLORS[0], label="truth")
axes[1].plot(grid, net.predict(grid[:, None]), color=e5.COLORS[2], label="network")
axes[1].set_title("Two hidden layers of 32 units")
axes[1].legend()
fig.tight_layout()
plt.show()

## Part 3. Optimisers

Adam is the default for a reason, but it is worth seeing what it improves on. Plain stochastic gradient descent takes a step proportional to the gradient. Momentum accumulates a velocity, damping oscillations across narrow valleys. Adam additionally scales each coordinate by a running estimate of its gradient magnitude.

In [ ]:
def train_simple(X, y, optimiser, n_epochs=300, learning_rate=0.01, batch_size=32, seed=4):
    """Train the same architecture with a chosen first-order optimiser."""
    net = NeuralNetwork([1, 32, 32, 1], seed=seed)
    velocity_W = [np.zeros_like(W) for W in net.W]
    velocity_b = [np.zeros_like(b) for b in net.b]
    losses = []

    for _ in range(n_epochs):
        order = net.generator.permutation(len(y))
        for start in range(0, len(y), batch_size):
            idx = order[start:start + batch_size]
            grads_W, grads_b = net.gradients(X[idx], y[idx])
            for layer in range(len(net.W)):
                if optimiser == "sgd":
                    net.W[layer] -= learning_rate * grads_W[layer]
                    net.b[layer] -= learning_rate * grads_b[layer]
                elif optimiser == "momentum":
                    velocity_W[layer] = 0.9 * velocity_W[layer] - learning_rate * grads_W[layer]
                    velocity_b[layer] = 0.9 * velocity_b[layer] - learning_rate * grads_b[layer]
                    net.W[layer] += velocity_W[layer]
                    net.b[layer] += velocity_b[layer]
        losses.append(net.loss(X, y))
    return np.array(losses)


adam_net = NeuralNetwork([1, 32, 32, 1], seed=4)
adam_losses = adam_net.fit(X_1d, y, n_epochs=300, learning_rate=0.01)["train"]

fig, ax = plt.subplots()
ax.plot(train_simple(X_1d, y, "sgd"), label="stochastic gradient descent")
ax.plot(train_simple(X_1d, y, "momentum"), label="momentum")
ax.plot(adam_losses, label="Adam")
ax.set_yscale("log")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("Same architecture, same learning rate, three optimisers")
ax.legend()
plt.show()

## Part 4. Regularisation

A network with thousands of parameters fitted to 200 observations will interpolate the noise unless something prevents it. Three devices do the preventing, and they are not interchangeable.

In [ ]:
order = rng.permutation(n)
train_idx, val_idx = order[:140], order[140:]
X_train, y_train = X_1d[train_idx], y[train_idx]
X_val, y_val = X_1d[val_idx], y[val_idx]

configurations = {
    "unregularised": {"weight_decay": 0.0, "dropout": 0.0},
    "weight decay": {"weight_decay": 1e-3, "dropout": 0.0},
    "dropout 0.2": {"weight_decay": 0.0, "dropout": 0.2},
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
rows = []

for name, config in configurations.items():
    net_r = NeuralNetwork([1, 64, 64, 1], seed=5, **config)
    hist = net_r.fit(X_train, y_train, n_epochs=500, learning_rate=0.01,
                     X_val=X_val, y_val=y_val)
    axes[0].plot(hist["validation"], label=name)
    axes[1].plot(grid, net_r.predict(grid[:, None]), label=name)
    rows.append({"configuration": name,
                 "final training loss": hist["train"][-1],
                 "final validation loss": hist["validation"][-1]})

axes[0].set_yscale("log")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("validation loss")
axes[0].set_title("Validation loss during training")
axes[0].legend(fontsize=9)

axes[1].scatter(x, y, s=10, color=e5.COLORS[6], alpha=0.4)
axes[1].plot(grid, f_target(grid), color="black", ls="--", label="truth")
axes[1].set_title("Fitted functions")
axes[1].legend(fontsize=9)
fig.tight_layout()
plt.show()

print(pd.DataFrame(rows).set_index("configuration").round(4))

### Exercise 2

Early stopping is the cheapest regulariser available and the one most often forgotten.

1. Train an unregularised network with `patience=40` and a validation set, using the `fit` method's early stopping arguments.
2. Record the epoch at which it stopped and the validation loss there.
3. Compare with training the same network for the full 500 epochs.

Explain, in terms of the path the optimiser takes through parameter space, why stopping early acts like a penalty on the size of the weights.

In [ ]:
net_early = NeuralNetwork([1, 64, 64, 1], seed=5)  #@keep
hist_early = net_early.fit(X_train, y_train, n_epochs=500, learning_rate=0.01,  #@keep
                           X_val=X_val, y_val=y_val, patience=40)  #@keep

net_long = NeuralNetwork([1, 64, 64, 1], seed=5)
hist_long = net_long.fit(X_train, y_train, n_epochs=500, learning_rate=0.01,
                         X_val=X_val, y_val=y_val)

print(f"early stopping halted at epoch {hist_early.get('stopped_at', 500)}")
print(f"validation loss, early stopped : {net_early.loss(X_val, y_val):.4f}")
print(f"validation loss, full training : {net_long.loss(X_val, y_val):.4f}")

norm_early = sum(np.sum(W ** 2) for W in net_early.W)
norm_long = sum(np.sum(W ** 2) for W in net_long.W)
print(f"\nsquared weight norm, early stopped : {norm_early:.2f}")
print(f"squared weight norm, full training : {norm_long:.2f}")

## Part 5. Overparameterisation and double descent

Lecture 3 established that test error is U-shaped in model complexity. Modern practice fits models with more parameters than observations and they work anyway. Both statements are true, and the resolution is that the U-shaped curve describes only the region up to the interpolation threshold.

The cleanest demonstration uses random features, which are a network with fixed inner weights, so the whole experiment is a sequence of linear regressions. When the number of features exceeds the number of observations, we take the minimum norm solution, which is what gradient descent from zero converges to.

In [ ]:
def random_feature_experiment(n_train=60, n_test=1000, widths=None, seed=0):
    """Test error of minimum-norm random feature regression across model sizes."""
    gen = np.random.default_rng(seed)
    widths = widths or [2, 5, 10, 20, 40, 50, 55, 58, 60, 62, 65, 70, 80, 120, 200, 400, 800]

    x_tr = gen.uniform(-3, 3, n_train)
    y_tr = f_target(x_tr) + 0.25 * gen.normal(size=n_train)
    x_te = gen.uniform(-3, 3, n_test)
    y_te = f_target(x_te) + 0.25 * gen.normal(size=n_test)

    errors = []
    for M in widths:
        W = gen.normal(scale=1.5, size=M)
        b = gen.normal(scale=2.0, size=M)
        H_tr = relu(x_tr[:, None] * W + b)
        H_te = relu(x_te[:, None] * W + b)

        coef = np.linalg.pinv(H_tr) @ y_tr          # minimum norm solution
        errors.append(e5.mse(y_te, H_te @ coef))
    return np.array(widths), np.array(errors)


widths, errors = random_feature_experiment(seed=1)
all_errors = np.array([random_feature_experiment(seed=s)[1] for s in range(20)])

fig, ax = plt.subplots()
ax.plot(widths, np.median(all_errors, axis=0), marker="o", ms=4)
ax.axvline(60, color=e5.COLORS[2], ls="--", label="interpolation threshold, $M = n$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of random features $M$")
ax.set_ylabel("median test mean squared error")
ax.set_title("Double descent")
ax.legend()
plt.show()

Error falls, then rises to a spike exactly where the number of features equals the number of observations, then falls again and keeps falling. At the threshold there is exactly one interpolating solution and it is a terrible one. Beyond it there are infinitely many, and the minimum norm choice among them is well behaved.

Nothing here contradicts Lecture 3. The classical analysis holds on the left of the spike. What it does not anticipate is that the implicit regularisation of the minimum norm solution takes over on the right.

### Exercise 3

Explicit regularisation should remove the spike, since the pathology comes from having exactly one interpolating solution.

1. Modify the experiment to fit ridge regression with a small penalty, $\lambda = 10^{-3}$, instead of the minimum norm solution.
2. Plot the ridge curve alongside the original.
3. Does the peak disappear, and is the overparameterised region still better than the best underparameterised model?

In [ ]:
def ridge_feature_experiment(lam, n_train=60, n_test=1000, seed=0):  #@keep
    """Random feature regression with an explicit ridge penalty."""  #@keep
    gen = np.random.default_rng(seed)
    widths = [2, 5, 10, 20, 40, 50, 55, 58, 60, 62, 65, 70, 80, 120, 200, 400, 800]

    x_tr = gen.uniform(-3, 3, n_train)
    y_tr = f_target(x_tr) + 0.25 * gen.normal(size=n_train)
    x_te = gen.uniform(-3, 3, n_test)
    y_te = f_target(x_te) + 0.25 * gen.normal(size=n_test)

    errors = []
    for M in widths:
        W = gen.normal(scale=1.5, size=M)
        b = gen.normal(scale=2.0, size=M)
        H_tr = relu(x_tr[:, None] * W + b)
        H_te = relu(x_te[:, None] * W + b)
        coef = np.linalg.solve(H_tr.T @ H_tr + lam * np.eye(M), H_tr.T @ y_tr)
        errors.append(e5.mse(y_te, H_te @ coef))
    return np.array(widths), np.array(errors)


ridge_errors = np.array([ridge_feature_experiment(1e-3, seed=s)[1] for s in range(20)])

fig, ax = plt.subplots()
ax.plot(widths, np.median(all_errors, axis=0), marker="o", ms=4, label="minimum norm")
ax.plot(widths, np.median(ridge_errors, axis=0), marker="s", ms=4,
        label="ridge, $\\lambda = 10^{-3}$")
ax.axvline(60, color=e5.COLORS[6], ls="--")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("number of random features $M$")
ax.set_ylabel("median test MSE")
ax.legend()
plt.show()

### Exercise 4

Compare your implementation against the standard one, which is a useful habit before trusting anything you wrote yourself.

1. Fit `sklearn.neural_network.MLPRegressor` with two hidden layers of 32 units, ReLU activation and the Adam solver, to the same data.
2. Compare the fitted function and the test error with your network.
3. Time both. Where does the difference come from?

In [ ]:
from sklearn.neural_network import MLPRegressor  #@keep
import time  #@keep

x_test = np.sort(rng.uniform(-3, 3, 500))
y_test = f_target(x_test) + noise * rng.normal(size=500)

start = time.time()
ours = NeuralNetwork([1, 32, 32, 1], seed=7)
ours.fit(X_1d, y, n_epochs=400, learning_rate=0.01)
time_ours = time.time() - start

start = time.time()
theirs = MLPRegressor(hidden_layer_sizes=(32, 32), activation="relu", solver="adam",
                      max_iter=400, learning_rate_init=0.01, random_state=0)
theirs.fit(X_1d, y)
time_theirs = time.time() - start

print(pd.DataFrame({
    "implementation": ["ours", "scikit-learn"],
    "test MSE": [e5.mse(y_test, ours.predict(x_test[:, None])),
                 e5.mse(y_test, theirs.predict(x_test[:, None]))],
    "seconds": [time_ours, time_theirs],
}).set_index("implementation").round(4))

fig, ax = plt.subplots()
ax.scatter(x, y, s=12, color=e5.COLORS[6], alpha=0.4)
ax.plot(grid, f_target(grid), color="black", ls="--", label="truth")
ax.plot(grid, ours.predict(grid[:, None]), color=e5.COLORS[0], label="ours")
ax.plot(grid, theirs.predict(grid[:, None]), color=e5.COLORS[2], label="scikit-learn")
ax.legend()
plt.show()

## Take-home challenges

1. **Width against depth.** Holding the total parameter count roughly fixed, compare a wide shallow network with a narrow deep one on the target function of this lab. Which does better, and does the answer change if the target is a composition of simple functions rather than a sine?

2. **Initialisation matters.** Replace the He initialisation, which scales by $\sqrt{2/n_{\text{in}}}$, with draws from $N(0, 1)$ and with all zeros. Train each and plot the loss curves. Explain why the all-zeros network never learns anything.

3. **Classification head.** Modify the network to output a probability through a sigmoid and to minimise cross-entropy rather than squared error. Derive the gradient of the new loss, which is simpler than you might expect, and test it on the two-class problem from Lab 2.

4. **Batch size.** Train with batch sizes 1, 8, 32 and the full sample. Plot loss against wall clock time rather than epochs. Which is fastest to a given loss, and why do very small batches help generalisation?

---

**Next**: Lab 8 takes this network to macroeconomic and financial data, adds recurrence for sequences, and asks whether any of it beats a well-tuned linear model.